In [ ]:
# %% Deep learning - Section 24.218
#    Code challenge 39: sine wave extrapolation
#
#    1) Start from code from video 24.217
#    2) Replace the alternating sequence with a sine function (see video)
#    3) Change the seq_length parameter appropriately
#    4) Test on new data with different frequency (see video)
#    5) Test on new data with different function (see video)
#    6) Test long-term extrapolation up to 2N datapoints

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.

In [1]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2
import torchvision
import torchvision.transforms as T
import torch.nn.utils         as utils
import random

from torch.utils.data                 import DataLoader,TensorDataset,Dataset,Subset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from matplotlib.gridspec              import GridSpec
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [ ]:
# %% Generate data

# Data
f    = 30
N    = 500
t    = torch.linspace(0,f*np.pi,N)
data = torch.sin(t + torch.cos(t))

# Plot
phi = (1 + np.sqrt(5)) / 2
plt.figure(figsize=(phi*6,6))

plt.plot([-1,N+1],[0,0],'--',color=[.8,.8,.8])
plt.plot(data.detach().cpu().numpy(),'ks-',markerfacecolor='w')
plt.xlim([-1,N+1])
plt.title('Some data')

plt.savefig('figure42_code_challenge_39.png')
plt.show()
files.download('figure42_code_challenge_39.png')


In [3]:
# %% RNN model class

class RNN(nn.Module):
    def __init__(self,input_size,num_hidden,num_layers):
        super().__init__()

        # RNN layer(s) and output
        self.rnn = nn.RNN(input_size,num_hidden,num_layers)
        self.out = nn.Linear(num_hidden,1)

    def forward(self,x,h):

        # Pass through RNN layers
        y,hidden = self.rnn(x,h)
        hidden   = hidden.detach()

        # Pass the RNN output through the fc output layer
        o = self.out(y)

        return o,hidden


In [ ]:
# %% Model's parameters

# Parameters
input_size =  1   # The data "channels"
num_hidden =  5   # Breadth of model (number of units in hidden layers)
num_layers =  1   # Depth of model (number of hidden layers)
seq_length =  50  # Number of datapoints used for learning in each segment
batch_size =  1   # (training code is hard-coded to organize data into batchsize=1)

# create an instance of the model and inspect
net = RNN(input_size,num_hidden,num_layers)

X   = torch.rand(seq_length,batch_size,input_size)
y,h = net(X,None)

# Note one output per sequence element (y.shape); generally, we take the final
# output to force a "many-to-one" design
print(X.shape)
print(y.shape)
print(h.shape)


In [ ]:
# %% Test the model on some data

# Data (transform into tensor with .view())
some_data = torch.tensor(data[:seq_length]).view(seq_length,1,1)
y = net(some_data,None)

# Grab final predicted value from the output (first element of tuple output of net)
final_value = y[0][-1]

# Loss (MSE is fine here even though binary CE is more appropriate)
loss_fun = nn.MSELoss()
loss_fun(final_value,torch.tensor(data[seq_length]).view(1,1))


In [ ]:
# %% Train model

# Epochs
num_epochs = 30

# New model and optimizer instance (SGD often used for standard RNNs)
net       = RNN(input_size,num_hidden,num_layers)
optimizer = torch.optim.SGD(net.parameters(),lr=.001)

# Preallocate losses and accuracy
losses        = np.zeros(num_epochs)
sign_accuracy = np.zeros(num_epochs)

# Loop
for epoch_i in range(num_epochs):

    # Loop over data segments (reset the hidden state on each epoch)
    seg_los      = []
    seg_acc      = []
    hidden_state = None

    for time_i in range(N-seq_length):

        # Grab data snippet (conceptually, x is size [1,9], y is size [1,1])
        X = data[time_i:time_i+seq_length].view(seq_length,1,1)
        y = data[time_i+seq_length].view(1,1)

        # Forward propagation and loss (compare final value of output)
        yHat,hidden_state = net(X,hidden_state)
        final_value       = yHat[-1]
        loss              = loss_fun(final_value,y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Loss from current segment
        seg_los.append(loss.item())

        # Sign accuracy
        true_sign = np.sign(torch.squeeze(y).numpy())
        pred_sign = np.sign(torch.squeeze(final_value).detach().numpy())
        accuracy  = 100*(true_sign==pred_sign)
        seg_acc.append(accuracy)

    # Average losses from current epoch
    losses[epoch_i]       = np.mean(seg_los)
    sign_accuracy[epoch_i] = np.mean(seg_acc)

    msg = f'Finished epoch {epoch_i+1}/{num_epochs}'
    sys.stdout.write('\r' + msg)


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,2,figsize=(1.5*phi*6,6))

ax[0].plot(losses,'s-')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss')
ax[0].set_title('Model loss')

ax[1].plot(sign_accuracy,'m^-',markerfacecolor='g',markersize=10)
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Accuracy')
ax[1].set_title(f'Sign accuracy (final accuracy: {sign_accuracy[-1]:.2f}%)')

plt.savefig('figure43_code_challenge_39.png')
plt.show()
files.download('figure43_code_challenge_39.png')


In [ ]:
# %% Test the model on the same data

# Initialize hidden state
h = np.zeros((N,num_hidden))

# Initialize predicted values (either zeros or nans)
yHat    = np.zeros(N)
yHat[:] = np.nan
hh      = None

# Loop over time segments
for time_i in range(N-seq_length):

    # Grab data snippet
    X = data[time_i:time_i+seq_length].view(seq_length,1,1)

    # Forward propagation and loss (also extract the hidden states)
    yy,hh = net(X,hh)
    yHat[time_i+seq_length] = yy[-1]
    h[time_i+seq_length,:] = hh.detach()

# Compute sign-accuracy (only pick seq_length vals because the first n elements
# cannot be predicted)
true_sign     = np.sign(data.numpy())
pred_sign     = np.sign(yHat)
sign_accuracy = 100*np.mean(true_sign[seq_length:]==pred_sign[seq_length:])


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,3,figsize=(1.5*phi*6,6))

ax[0].plot(data,'bs-',label='Actual data')
ax[0].plot(yHat,'ro-',label='Predicted')
ax[0].set_ylim([-1.1,1.1])
ax[0].set_title(f'Sign accuracy (final accuracy: {sign_accuracy:.2f}%)')
ax[0].legend()

ax[1].plot(data-yHat,'k^')
ax[1].set_ylim([-1.1,1.1])

ax[2].plot(data[seq_length:],yHat[seq_length:],'mo')
ax[2].set_xlabel('Real data')
ax[2].set_ylabel('Predicted data')
r = np.corrcoef(data[seq_length:],yHat[seq_length:])
ax[2].set_title(f"r={r[0,1]:.2f} (but Simpson's paradox!)")

plt.suptitle('Performance on training data',fontweight='bold',fontsize=20,y=1.1)
plt.tight_layout()

plt.savefig('figure44_code_challenge_39.png')
plt.show()
files.download('figure44_code_challenge_39.png')


In [ ]:
# %% Test on new data (option 1)

# Create new data
f        = 10
N        = 500
t        = torch.linspace(0,f*np.pi,N)
new_data = torch.sin(t + torch.cos(t))

# Plot
phi = (1 + np.sqrt(5)) / 2
plt.figure(figsize=(phi*6,6))

plt.plot([-1,N+1],[0,0],'--',color=[.8,.8,.8])
plt.plot(new_data.detach().cpu().numpy(),'ks-',markerfacecolor='w')
plt.xlim([-1,N+1])
plt.title('Some new data')

plt.savefig('figure45_code_challenge_39.png')
plt.show()
files.download('figure45_code_challenge_39.png')

# Test the network (no learning here)
h    = np.zeros((N,num_hidden))
yHat = np.zeros(N)
hh   = None

for time_i in range(N-seq_length):

    # Grab data snippet
    X = new_data[time_i:time_i+seq_length].view(seq_length,1,1)

    # Forward propagation and loss
    yy,hh = net(X,hh)
    yHat[time_i+seq_length] = yy[-1]
    h[time_i+seq_length,:] = hh.detach()

# Compute sign-accuracy
true_sign     = np.sign(new_data.numpy())
pred_sign     = np.sign(yHat)
sign_accuracy = 100*np.mean(true_sign[seq_length:]==pred_sign[seq_length:])


In [ ]:
# %% Test on new data (option 2)

# Create new data
f        = 30
N        = 500
t        = torch.linspace(0,f*np.pi,N)
new_data = torch.sin(t + torch.sin(t))

# Plot
phi = (1 + np.sqrt(5)) / 2
plt.figure(figsize=(phi*6,6))

plt.plot([-1,N+1],[0,0],'--',color=[.8,.8,.8])
plt.plot(new_data.detach().cpu().numpy(),'ks-',markerfacecolor='w')
plt.xlim([-1,N+1])
plt.title('Some new data')

plt.savefig('figure47_code_challenge_39.png')
plt.show()
files.download('figure47_code_challenge_39.png')

# Test the network (no learning here)
h    = np.zeros((N,num_hidden))
yHat = np.zeros(N)
hh   = None

for time_i in range(N-seq_length):

    # Grab data snippet
    X = new_data[time_i:time_i+seq_length].view(seq_length,1,1)

    # Forward propagation and loss
    yy,hh = net(X,hh)
    yHat[time_i+seq_length] = yy[-1]
    h[time_i+seq_length,:] = hh.detach()

# Compute sign-accuracy
true_sign     = np.sign(new_data.numpy())
pred_sign     = np.sign(yHat)
sign_accuracy = 100*np.mean(true_sign[seq_length:]==pred_sign[seq_length:])


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,3,figsize=(1.5*phi*6,6))

ax[0].plot(new_data,'bs-',label='Actual data')
ax[0].plot(yHat,'ro-',label='Predicted')
ax[0].set_ylim([-1.1,1.1])
ax[0].legend()
ax[0].set_title(f'Sign accuracy (final accuracy: {sign_accuracy:.2f}%)')

ax[1].plot(new_data-yHat,'k^')
ax[1].set_ylim([-1.1,1.1])
ax[1].set_title(f'Sign accuracy = {sign_accuracy:.2f}%')

ax[2].plot(new_data[seq_length:],yHat[seq_length:],'mo')
ax[2].set_xlabel('Real data')
ax[2].set_ylabel('Predicted data')
r = np.corrcoef(new_data[seq_length:],yHat[seq_length:])
ax[2].set_title(f"r={r[0,1]:.2f} (but Simpson's paradox!)")

plt.suptitle('Performance on unseen test data',fontweight='bold',fontsize=20,y=1.1)
plt.tight_layout()

plt.savefig('figure46_code_challenge_39.png')
plt.show()
files.download('figure46_code_challenge_39.png')


In [10]:
# %% Extrapolate using original data

# Generate 2N signal
yHat     = torch.zeros(2*N)
yHat[:N] = data
hh       = None

# Try longer sequences (original s = 50)
seq_length = 150

for time_i in range(2*N-seq_length):

    # Grab data snippet
    X = yHat[time_i:time_i+seq_length].view(seq_length,1,1)

    # Forward propagation and loss
    yy,hh = net(X,hh)
    yHat[time_i+seq_length] = yy[-1]


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig = plt.figure(figsize=(phi*6,6))

plt.plot(data,'bs-',label='Actual data',markersize=3,alpha=0.8)
plt.plot(yHat.detach(),'ro-',label='Predicted',markersize=3,alpha=0.8)
plt.axvline(x=seq_length,color=[.3,.3,.3],linestyle='--',label='Sequence length')
plt.ylim([-1.1,1.1])
plt.legend()

plt.title(f'Performance on extrapolated data\n(sequence length = {seq_length})')

plt.savefig('figure49_code_challenge_39.png')
plt.show()
files.download('figure49_code_challenge_39.png')


In [ ]:
# %% Exercise 1
#    Extrapolation was awful! Maybe the model is too simple? Try increasing the sequence length and the number
#    of hidden units. Does that help the extrapolation?

# Just inreasing the sequence lengh helps only in the sense that the sequence is
# accurately extrapolated up to the sequence length itself, after that value is
# passed, the performance degrades very fast no matter the value of seq_length.
# Using more hidden layers does help; 3 layers improves considerably, the
# predicted sequence now is not perfect but at least oscillates, with 5 layers
# there seems to be a marginal improvement


In [ ]:
# %% Exercise 1
#    Continue ...

# Set up with more hidden layers

input_size =  1
num_hidden =  5
num_layers =  5
seq_length =  50
batch_size =  1

net = RNN(input_size,num_hidden,num_layers)

X   = torch.rand(seq_length,batch_size,input_size)
y,h = net(X,None)

print(X.shape)
print(y.shape)
print(h.shape)
